In [ ]:
import json
from collections import defaultdict
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

# Path to the chunks file
chunks_path = '../data/processed/danish_dataset_7_no_chain/merged/chunks.jsonl'

# Load chunks and group by doc_id
doc_chunks = defaultdict(list)
with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        chunk_id = entry['chunk_id']
        chunk_text = entry['chunk']
        doc_id = chunk_id.split('_')[0]
        doc_chunks[doc_id].append(chunk_text)

print(f"Loaded {len(doc_chunks)} documents.")
for doc_id, chunks in doc_chunks.items():
    print(f"{doc_id}: {len(chunks)} chunks")


/home/mpj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: invalid literal for int() with base 10: 'stk1'

In [10]:
# Load the sentence transformer model
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name)

# Compute average embedding for each document
doc_embeddings = {}
for doc_id, chunks in tqdm(doc_chunks.items()):
    chunk_embs = model.encode(chunks, show_progress_bar=False)
    avg_emb = np.mean(chunk_embs, axis=0)
    doc_embeddings[doc_id] = avg_emb

print(f"Computed embeddings for {len(doc_embeddings)} documents.")
embedding_dim = next(iter(doc_embeddings.values())).shape[0]
print(f"Embedding dimension: {embedding_dim}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11458.12it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 7/7 [03:06<00:00, 26.65s/it]

Computed embeddings for 7 documents.
Embedding dimension: 384


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Prepare doc_id list and embedding matrix
doc_ids = sorted(doc_embeddings.keys())
emb_matrix = np.stack([doc_embeddings[doc_id] for doc_id in doc_ids])

# Compute cosine similarity matrix
sim_matrix = cosine_similarity(emb_matrix)

# Create a DataFrame for pretty display
sim_df = pd.DataFrame(sim_matrix, index=doc_ids, columns=doc_ids)

# Display the similarity table
sim_df.round(3)

,aarsregnskabsloven,almenboligloven,barnetslov,erhvervsfondsloven,miljøbeskyttelsesloven,selskabsloven,serviceloven
aarsregnskabsloven,1.000,0.824,0.729,0.921,0.845,0.924,0.818
almenboligloven,0.824,1.000,0.831,0.847,0.850,0.826,0.897
barnetslov,0.729,0.831,1.000,0.758,0.757,0.743,0.964
erhvervsfondsloven,0.921,0.847,0.758,1.000,0.844,0.960,0.835
miljøbeskyttelsesloven,0.845,0.850,0.757,0.844,1.000,0.832,0.864
selskabsloven,0.924,0.826,0.743,0.960,0.832,1.000,0.822
serviceloven,0.818,0.897,0.964,0.835,0.864,0.822,1.000
